<a href="https://colab.research.google.com/github/ketanp23/deeplearningclass/blob/main/DL_Lab7_Decision_Boundaries_and_Features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DL Lab 7 · Decision Boundaries & Feature Learning

> **Deep Learning foundations — Lab 7 (Part II).** A deeper companion to **Lab 2 (XOR)**. Run top to bottom
> (Runtime → Run all), read the plain-English notes, and finish the **Your turn 🧪** cell. Built from scratch with
> NumPy so nothing is hidden.

### In plain English
Lab 2 showed that a hidden layer solves XOR by **bending the space** until the classes are linearly separable. Here we
push that idea onto harder, curved datasets — **moons** and **circles** — visualize the boundaries a network learns,
peek at the **hidden feature space** where the bending happens, and see how **more hidden units** buy more curvature.

In [1]:
import numpy as np, matplotlib.pyplot as plt
VIOLET,CYAN,AMBER="#6C5CE7","#22D3EE","#F59E0B"

def make_moons(n=240, noise=0.18, seed=0):
    rng=np.random.default_rng(seed); n2=n//2; t=np.linspace(0,np.pi,n2)
    a=np.c_[np.cos(t), np.sin(t)]; b=np.c_[1-np.cos(t), 0.5-np.sin(t)]
    X=np.vstack([a,b]).astype(float); y=np.r_[np.zeros(n2),np.ones(n2)]
    return X+rng.normal(0,noise,X.shape), y
def make_circles(n=240, noise=0.10, factor=0.45, seed=0):
    rng=np.random.default_rng(seed); n2=n//2; t=np.linspace(0,2*np.pi,n2,endpoint=False)
    out=np.c_[np.cos(t),np.sin(t)]; inn=out*factor
    X=np.vstack([out,inn]).astype(float); y=np.r_[np.zeros(n2),np.ones(n2)]
    return X+rng.normal(0,noise,X.shape), y

## 1 · A small MLP trainer (from scratch)

In [2]:
class MLP:
    def __init__(self, sizes, act='tanh', seed=0):
        rng=np.random.default_rng(seed); self.W=[]; self.b=[]; self.act=act
        for i in range(len(sizes)-1):
            sc=np.sqrt(2.0/sizes[i]) if act=='relu' else np.sqrt(1.0/sizes[i])
            self.W.append(rng.standard_normal((sizes[i],sizes[i+1]))*sc)
            self.b.append(np.zeros(sizes[i+1]))
    def _f(self,z):  return np.tanh(z) if self.act=='tanh' else np.maximum(0,z)
    def _df(self,a): return (1-a**2) if self.act=='tanh' else (a>0).astype(float)
    def forward(self,X):
        self.a=[X]; a=X; L=len(self.W)
        for i in range(L):
            z=a@self.W[i]+self.b[i]
            a=1/(1+np.exp(-z)) if i==L-1 else self._f(z)
            self.a.append(a)
        return a
    def fit(self,X,y,epochs=4000,lr=0.7):
        y=y.reshape(-1,1); n=len(X); hist=[]
        for e in range(epochs):
            out=self.forward(X); hist.append(np.mean((out-y)**2))
            d=(out-y)*out*(1-out)
            for i in range(len(self.W)-1,-1,-1):
                gW=self.a[i].T@d/n; gb=d.mean(0)
                if i>0: d=(d@self.W[i].T)*self._df(self.a[i])
                self.W[i]-=lr*gW; self.b[i]-=lr*gb
        return hist
    def prob(self,X): return self.forward(X).ravel()
    def predict(self,X): return (self.prob(X)>0.5).astype(int)

def boundary(ax, model, X, y, title):
    xx,yy=np.meshgrid(np.linspace(X[:,0].min()-.5,X[:,0].max()+.5,200),
                      np.linspace(X[:,1].min()-.5,X[:,1].max()+.5,200))
    Z=model.prob(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx,yy,Z,levels=20,cmap="RdBu_r",alpha=.8)
    ax.scatter(X[:,0],X[:,1],c=y,cmap="RdBu_r",edgecolors="k",s=18,linewidths=.4)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

## 2 · Learn curved boundaries

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4.6))
for a,(name,(X,y)) in zip(ax, {"moons":make_moons(),"circles":make_circles()}.items()):
    m=MLP([2,16,16,1],act='tanh',seed=1); m.fit(X,y,epochs=4000,lr=0.7)
    acc=(m.predict(X)==y).mean()
    boundary(a,m,X,y,f"{name} — accuracy {acc:.2f}")
plt.tight_layout(); plt.show()

## 3 · Where the bending happens: the hidden feature space
Give the network exactly **2 units** in its last hidden layer, so we can plot what it hands to the output neuron. The
output is just a straight line — but in *these learned coordinates*, the tangled classes have been pulled apart.

In [ ]:
X,y=make_moons(noise=0.15, seed=2)
m=MLP([2,16,2,1],act='tanh',seed=3); m.fit(X,y,epochs=5000,lr=0.7)
m.forward(X); H=m.a[-2]                       # activations of the 2-unit hidden layer
fig,ax=plt.subplots(1,2,figsize=(11,4.4))
ax[0].scatter(X[:,0],X[:,1],c=y,cmap="RdBu_r",edgecolors="k",s=18); ax[0].set_title("input space (tangled)")
ax[1].scatter(H[:,0],H[:,1],c=y,cmap="RdBu_r",edgecolors="k",s=18); ax[1].set_title("last hidden space (untangled → linearly separable)")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()
print("accuracy:", (m.predict(X)==y).mean())

## 4 · More hidden units → more curvature

In [ ]:
X,y=make_circles(noise=0.08, seed=4)
fig,ax=plt.subplots(1,3,figsize=(13,4.2))
for a,H in zip(ax,[1,4,16]):
    m=MLP([2,H,1],act='tanh',seed=5); m.fit(X,y,epochs=4000,lr=0.8)
    boundary(a,m,X,y,f"{H} hidden unit(s) — acc {(m.predict(X)==y).mean():.2f}")
plt.tight_layout(); plt.show()
print("1 unit can only draw a line; a handful already wrap the inner circle.")

## Your turn 🧪
1. Make a **spiral** dataset and try to separate it. How many units/layers does it need?
2. Switch the hidden activation to `'relu'`. Do the boundaries become more *angular* (piecewise-linear)?
3. Increase `noise` in `make_moons` until the network can no longer reach high accuracy. What is it hitting?

In [ ]:
# Your turn: relu gives piecewise-linear (angular) boundaries
X,y=make_moons(seed=6)
m=MLP([2,16,16,1],act='relu',seed=7); m.fit(X,y,epochs=4000,lr=0.3)
print("relu moons accuracy:", (m.predict(X)==y).mean())